In [0]:
%sql
select is_active From workspace.silver.crm_cust_info

In [0]:
%sql
update workspace.bronze.crm_cust_info set cst_gndr = 'F' where cst_id = 11000

In [0]:
%sql
select * from workspace.bronze.crm_cust_info where cst_id = 11000

In [0]:
%sql
insert into workspace.bronze.crm_cust_info (
    cst_id,
    cst_key,
    cst_firstname,
    cst_lastname,
    cst_marital_status,
    cst_gndr,
    cst_create_date
) 
values (
    99999,
    'AW01010101',
    'Ann',
    'Gonzaga',
    'M',
    'F',
    '2025-10-06'
)

In [0]:
%sql
select * from workspace.bronze.crm_cust_info where cst_id = 99999

In [0]:
from silver_config import transformation_config
from pyspark.sql import functions as F

def add_hash(df, compare_columns):

    return df.withColumn(
        "hash_code",
        F.sha2(
            F.concat_ws(
                "||",
                *[
                    F.coalesce(
                        F.col(column).cast("string"),
                        F.lit("")
                    )
                    for column in compare_columns
                ]
            ),
            256
        )
    )

# data from bronze layer --------------------
bronze_df = spark.sql(
            transformation_config[0]["transformation"]
        )

bronze_df = add_hash(
            bronze_df,
            transformation_config[0]["compare_columns"]
        )

filtered = (
    bronze_df.filter(
    F.col(transformation_config[0]["table_key"]) == 99999
    )
)

filtered.show()


# data from silver layer --------------------
silver_df = spark.sql(
            """
            select * from workspace.silver.crm_cust_info
            """
        )

silver_df = add_hash(
            silver_df,
            transformation_config[0]["compare_columns"]
            )

joined_df = (
        bronze_df.alias("b")
        .join(
            silver_df.alias("s"),
            F.col(f"b.{transformation_config[0]["table_key"]}") == F.col(f"s.{transformation_config[0]["table_key"]}"),
            "left"
        )
    )

filtereds = (
    joined_df.filter(
    F.col(f"b.{transformation_config[0]["table_key"]}") == 99999
    )
)

filtereds.show()

new_df = (
        joined_df
        .filter(
            F.col(f"s.{transformation_config[0]["table_key"]}").isNull()
        )
        .select("b.*")
        .withColumn("is_active", F.lit(True))
        .withColumn("date_activated", F.current_timestamp())
        .withColumn(
            "date_deactivated",
            F.lit(None).cast("timestamp")
        )
    )

new_df.show()





In [0]:
%sql
DESCRIBE TABLE workspace.silver.crm_cust_info;

In [0]:
%sql
select * from workspace.silver.crm_cust_info where cst_id = 11000

In [0]:
%sql
select * from workspace.bronze.crm_cust_info where cst_id = 11000

In [0]:
df = spark.sql("select * from workspace.silver.crm_prd_info")
df.printSchema()

In [0]:
from silver_config import transformation_config

for item in transformation_config:

    df = spark.table(item["target_table"])

    data_type = df.schema["date_deactivated"].dataType

    if data_type.typeName() == "void":
        print(f"{item['target_table']} -> VOID")
        spark.sql(f"""
                ALTER TABLE {item["target_table"]}
                ALTER COLUMN date_deactivated TYPE TIMESTAMP;
                  """)
    else:
        print(f"{item['target_table']} -> {data_type.typeName()}")

In [0]:
%sql

DESCRIBE workspace.silver.crm_cust_info;

In [0]:
%sql
select * from workspace.silver.crm_cust_info where cst_id = 99999 or cst_id = 11000
--delete from workspace.silver.crm_cust_info where cst_id = 99999

In [0]:
%sql
select * from workspace.silver.erp_loc_a101

In [0]:
from silver_config import transformation_config

for item in transformation_config:

    df = spark.sql(f"""
        select * From {item["target_table"]}""")
    
    df.show()
    #print(f"SUCCESS | TRUNCATE | TABLE NAME: {item["target_table"]}")

In [0]:
%sql
select * From workspace.bronze.erp_px_cat_g1v2

In [0]:
%sql
select * From workspace.silver.erp_px_cat_g1v2

In [0]:
%sql
select * From workspace.gold.dim_customers

In [0]:
%sql
select * from workspace.gold.fact_sales